In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Load Data
df = pd.read_csv("bot_detection_data.csv")

# Preprocessing: Feature Engineering
df['Created At'] = pd.to_datetime(df['Created At'])
df['account_age_days'] = (pd.Timestamp.now() - df['Created At']).dt.days

# Select features and target
features = ['Retweet Count', 'Mention Count', 'Follower Count', 'Verified', 'account_age_days']
X = df[features]
y = df['Bot Label']

# Handle categorical 'Verified' (boolean to int)
X['Verified'] = X['Verified'].astype(int)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing Pipeline
numeric_features = ['Retweet Count', 'Mention Count', 'Follower Count', 'account_age_days']
categorical_features = ['Verified']

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", "passthrough", categorical_features)
    ])

# Final Pipeline
clf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the model
clf_pipeline.fit(X_train, y_train)

# Predict and Evaluate
y_pred = clf_pipeline.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


C:\Users\Admin\AppData\Local\Temp\ipykernel_35192\2810657354.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Verified'] = X['Verified'].astype(int)


Classification Report:
               precision    recall  f1-score   support

           0       0.50      0.52      0.51      4968
           1       0.51      0.48      0.49      5032

    accuracy                           0.50     10000
   macro avg       0.50      0.50      0.50     10000
weighted avg       0.50      0.50      0.50     10000

Confusion Matrix:
 [[2606 2362]
 [2621 2411]]


In [3]:
import joblib

# After training:
joblib.dump(clf_pipeline, 'bot_detector_pipeline.pkl')

# Later, to load it back:
clf_pipeline = joblib.load('bot_detector_pipeline.pkl')


In [5]:
import pandas as pd

# Example new user data:
new_user = pd.DataFrame([{
    'Retweet Count': 12,
    'Mention Count': 3,
    'Follower Count': 4500,
    'Verified': False,
    'account_age_days': 365  # e.g., account is one year old
}])

# Make sure dtype matches (bool → int)
new_user['Verified'] = new_user['Verified'].astype(int)

# Predict
prediction = clf_pipeline.predict(new_user)
probability = clf_pipeline.predict_proba(new_user)[:,1]  # bot‐probability

print(f"Label: {prediction[0]}  (1=Bot, 0=Human)")
print(f"Bot Probability: {probability[0]:.2f}")


Label: 0  (1=Bot, 0=Human)
Bot Probability: 0.45
